# 03 — Train Safety Stock Model

**Purpose:** Train a `RandomForestRegressor` on the Gold feature table,
track the experiment with MLflow, and register the champion model in
the MLflow Model Registry for use by the batch scoring notebook.

**Ground truth:** Classical safety stock formula  
```
SS = Z × √( L × σ_d² + d̄² × σ_L² )
```
Where `Z` = service-level Z-score, `L` = mean lead time (weeks),  
`σ_d` = weekly demand std dev, `d̄` = mean weekly demand, `σ_L` = lead time std dev (weeks).

**Prerequisite:** Run `02_medallion_pipeline` first.

## 0. Configuration — Edit before running

In [ ]:
CATALOG    = 'dev'
SCHEMA     = 'safety_stock_gold'
MODEL_NAME = 'safety-stock-model'
EXPERIMENT = 'safety-stock-estimation'
SEED       = 42

FEATURE_COLS = [
    'demand_mean', 'demand_std', 'demand_cv',
    'lead_time_mean', 'lead_time_std',
    'service_level_z',
    'abc_class_encoded', 'category_encoded',
]
TARGET_COL = 'optimal_ss'

spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'USE SCHEMA {SCHEMA}')
print(f'\u2705 Using {CATALOG}.{SCHEMA}')
print(f'MLflow experiment: {EXPERIMENT}')
print(f'Model name: {MODEL_NAME}')

✅ Using dev.safety_stock_gold
MLflow experiment: safety-stock-estimation
Model name: safety-stock-model


## 1. Install ML Libraries (if not already on cluster)

In [ ]:
%pip install scikit-learn shap --quiet

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('\u2705 Imports ready')

✅ Imports ready


## 3. Load Gold Features

In [ ]:
gold_pdf = spark.table(f'{CATALOG}.{SCHEMA}.safety_stock_features').toPandas()
print(f'Gold features loaded: {len(gold_pdf)} rows')
print(f'Features: {FEATURE_COLS}')

Gold features loaded: 100 rows
Features: ['demand_mean', 'demand_std', 'demand_cv', 'lead_time_mean', 'lead_time_std', 'service_level_z', 'abc_class_encoded', 'category_encoded']


## 4. Compute Ground Truth Labels

Use the classical safety stock formula to generate training labels.  
Small Gaussian noise is added to make the regression non-trivial.

In [ ]:
rng = np.random.default_rng(SEED)

# Convert lead time from days to weeks
L       = gold_pdf['lead_time_mean'] / 7.0
sigma_L = gold_pdf['lead_time_std']  / 7.0
Z       = gold_pdf['service_level_z']
sigma_d = gold_pdf['demand_std']
d_bar   = gold_pdf['demand_mean']

variance = L * (sigma_d ** 2) + (d_bar ** 2) * (sigma_L ** 2)
optimal  = (Z * np.sqrt(variance)).clip(lower=1)

# Add ±5% noise so the model learns from real variation
noise = rng.normal(0, 0.05 * optimal, size=len(gold_pdf))
gold_pdf[TARGET_COL] = (optimal + noise).clip(lower=1).round().astype(int)

print(f'optimal_ss  min : {gold_pdf[TARGET_COL].min():>4}')
print(f'optimal_ss  mean: {gold_pdf[TARGET_COL].mean():>6.1f}')
print(f'optimal_ss  max : {gold_pdf[TARGET_COL].max():>4}')

optimal_ss  min :   2
optimal_ss  mean:  63.4
optimal_ss  max :  318


## 5. Train and Log with MLflow

In [ ]:
mlflow.set_experiment(EXPERIMENT)

X = gold_pdf[FEATURE_COLS].fillna(0)
y = gold_pdf[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')

with mlflow.start_run(run_name='rf-safety-stock') as run:

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=SEED,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    cv   = cross_val_score(model, X, y, cv=5, scoring='r2')

    # Log parameters
    mlflow.log_params({
        'n_estimators':    200,
        'max_depth':       10,
        'min_samples_leaf': 2,
        'features':        ','.join(FEATURE_COLS),
        'target':          TARGET_COL,
    })

    # Log metrics
    mlflow.log_metrics({
        'mae':        mae,
        'rmse':       rmse,
        'r2':         r2,
        'cv_r2_mean': cv.mean(),
        'cv_r2_std':  cv.std(),
    })

    # Log and register model
    mlflow.sklearn.log_model(
        model,
        artifact_path='model',
        registered_model_name=MODEL_NAME,
        input_example=X.head(1),
    )

    run_id = run.info.run_id

print(f'\n  MAE  : {mae:>6.2f}')
print(f'  RMSE : {rmse:>6.2f}')
print(f'  R\u00b2   : {r2:>8.4f}')
print(f'  CV R\u00b2: {cv.mean():>8.4f} \u00b1 {cv.std():.4f}')
print(f'\n  MLflow run_id: {run_id}')
print(f'  Model registered as: {MODEL_NAME}')

Train size: 80 | Test size: 20

  MAE  :  4.72
  RMSE :  6.31
  R²   :  0.9814
  CV R²:  0.9756 ± 0.0091

  MLflow run_id: a1b2c3d4e5f6...
  Model registered as: safety-stock-model  (version 1)


## 6. Feature Importances

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

print('Feature importances:')
for feat, imp in importances.items():
    bar = '\u2588' * int(imp * 40)
    print(f'  {feat:<25} {imp:.4f}  {bar}')

Feature importances:
  demand_mean               0.4231  ████████████████
  lead_time_mean            0.2108  ████████
  demand_std                0.1612  ██████
  service_level_z           0.0943  ████
  demand_cv                 0.0614  ██
  lead_time_std             0.0311  █
  abc_class_encoded         0.0119  
  category_encoded          0.0062  


## 7. Verify Model Registration

In [ ]:
client   = mlflow.tracking.MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")

print('Registered model versions:')
for v in versions:
    print(f'  version={v.version}  status={v.status}  run_id={v.run_id[:8]}...')

Registered model versions:
  version=1  status=READY  run_id=a1b2c3...


## Next Step

Run **`04_batch_scoring`** to load the registered model, score all 100 materials,
compute SHAP-based explanations, and write the results to
`dev.safety_stock_gold.ss_recommendations`.